             dim_customer

                  │

             customer_key

                  │

                  ▼

             fact_sales

                  ▲

                  │

              product_key

                  │
                  
             dim_product

This is a star schema.

## Read the three tables

In [0]:
from pyspark.sql import functions as F

In [0]:
sales = spark.table(
    "e2e_project.silver.crm_sales_details"
)

customers = spark.table(
    "e2e_project.gold.dim_customer"
)

products = spark.table(
    "e2e_project.gold.dim_product"
)

In [0]:
display(sales.limit(10))
display(customers.limit(10))
display(products.limit(10))

In [0]:
sales.printSchema()

## Understand how the joins work

The transformation is:

Source identifiers

        ↓

look up dimension

        ↓
        
Warehouse surrogate keys

That's one of the most important concepts in dimensional modeling.

## Before joining: make sure the dimensions are unique

check customers

In [0]:
display(
    customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

check products

In [0]:
display(
    products
    .groupBy("product_number")
    .count()
    .filter(F.col("count") > 1)
)

## Check customer matching


In [0]:
unmatched_customers = (
    sales
    .join(
        customers.select(
            "customer_id"
        ),
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Sales with unknown customer:",
    unmatched_customers.count()
)

display(unmatched_customers.limit(20))

## Check product matching

In [0]:
unmatched_products = (
    sales.alias("s")
    .join(
        products.select(
            "product_number"
        ).alias("p"),
        F.col("s.product_number") ==
        F.col("p.product_number"),
        "left_anti"
    )
)

print(
    "Sales with unknown product:",
    unmatched_products.count()
)

display(unmatched_products.limit(20))

## Create small dimension lookup DataFrames

In [0]:
customer_lookup = customers.select(
    "customer_id",
    "customer_key"
)

product_lookup = products.select(
    "product_number",
    "product_key"
)

customer_lookup = customers.select(
    "customer_id",
    "customer_key"
)

product_lookup = products.select(
    "product_number",
    "product_key"
)

## Join sales with customer dimension

In [0]:
sales_with_customer = (
    sales.alias("s")
    .join(
        customer_lookup.alias("c"),
        F.col("s.customer_id") ==
        F.col("c.customer_id"),
        "left"
    )
)

## Join with product dimension

In [0]:
sales_enriched = (
    sales_with_customer.alias("s")
    .join(
        product_lookup.alias("p"),
        F.col("s.product_number") ==
        F.col("p.product_number"),
        "left"
    )
)

## Build the final fact table

In [0]:
fact_sales = sales_enriched.select(

    F.col("s.order_number")
        .alias("order_number"),

    F.col("p.product_key")
        .alias("product_key"),

    F.col("s.customer_key")
        .alias("customer_key"),

    F.col("s.order_date")
        .alias("order_date"),

    F.col("s.ship_date")
        .alias("ship_date"),

    F.col("s.due_date")
        .alias("due_date"),

    F.col("s.sales_amount")
        .alias("sales_amount"),

    F.col("s.quantity")
        .alias("quantity"),

    F.col("s.price")
        .alias("price")
)

## Understand why we don't duplicate dimension attributes

Bad design: 

fact_sales


order_number

customer_name

customer_country

customer_gender

product_name

product_category

product_subcategory

sales_amount
...

All that information would repeat thousands of times.


Instead:


fact_sales


customer_key = 10

product_key  = 52

sales_amount = 120

That's exactly why dimensional modeling uses

 facts + dimensions.

## Validate your row count

In [0]:
print("Silver sales rows:", sales.count())
print("Gold fact rows:", fact_sales.count())

Silver sales rows

=

Gold fact rows

## Check missing foreign keys

These are called referential integrity checks.

We're essentially verifying:

Every customer_key in fact_sales
must exist in dim_customer

Every product_key in fact_sales
must exist in dim_product

In [0]:
display(
    fact_sales.filter(
        F.col("customer_key").isNull()
    )
)

In [0]:
display(
    fact_sales.filter(
        F.col("product_key").isNull()
    )
)

## Validate the sales calculation again

In [0]:
display(
    fact_sales.filter(
        F.abs(
            F.col("sales_amount") -
            (
                F.col("quantity") *
                F.col("price")
            )
        ) > 0.01
    )
)

## Check date relationships

Check shipping before order:

In [0]:
display(
    fact_sales.filter(
        F.col("ship_date") <
        F.col("order_date")
    )
)

Check due date before order:

In [0]:
display(
    fact_sales.filter(
        F.col("due_date") <
        F.col("order_date")
    )
)

## Check important NULLs

In [0]:
display(
    fact_sales.select(
        [
            F.sum(
                F.col(c).isNull().cast("int")
            ).alias(c)
            for c in fact_sales.columns
        ]
    )
)

## Save fact_sales

In [0]:
(
    fact_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "e2e_project.gold.fact_sales"
    )
)

In [0]:
%sql

SELECT *
FROM e2e_project.gold.fact_sales
LIMIT 20;

### Even though Delta tables don't automatically behave exactly like a traditional relational DB with enforced PK/FK constraints, this is the logical relationship of your warehouse.

## Test whether the star schema actually works

Which countries generate the most revenue?

In [0]:
%sql

SELECT
    c.country,
    SUM(f.sales_amount) AS total_revenue
FROM e2e_project.gold.fact_sales f
LEFT JOIN e2e_project.gold.dim_customer c
    ON f.customer_key = c.customer_key
GROUP BY c.country
ORDER BY total_revenue DESC;

Try product category

In [0]:
%sql

SELECT
    p.category,
    SUM(f.sales_amount) AS total_revenue
FROM e2e_project.gold.fact_sales f
LEFT JOIN e2e_project.gold.dim_product p
    ON f.product_key = p.product_key
GROUP BY p.category
ORDER BY total_revenue DESC;

monthly revenue

In [0]:
%sql

SELECT
    YEAR(order_date) AS year,
    MONTH(order_date) AS month,
    SUM(sales_amount) AS revenue
FROM e2e_project.gold.fact_sales
GROUP BY
    YEAR(order_date),
    MONTH(order_date)
ORDER BY
    year,
    month;

Your complete pipeline now